# 03 — Extract Betas & Voxelwise Regression

This notebook:
1. Extracts NSD single-trial beta estimates for selected images
2. Runs voxelwise GLM: BOLD ~ load + abstract + load×abstract + nuisance
3. Computes group-level t-maps with FDR correction
4. Visualizes beta maps

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

sns.set_style('whitegrid')
sns.set_context('notebook')
%matplotlib inline

## Configuration

In [ ]:
# ======================== EDIT THESE PATHS ========================
NSD_ROOT = '/path/to/nsd'                         # NSD data root
SPACE = 'fsaverage'                                # 'fsaverage' or 'volume'
# =================================================================

SCORES_FILE = 'data/nsd_image_scores.csv'
SELECTED_IDS = 'data/selected_image_ids.csv'
BETA_DIR = 'results/beta_maps/'
SUBJECTS = [f'sub-{i:02d}' for i in range(1, 9)]
FDR_Q = 0.05

## Step 1: Extract Betas

In [ ]:
from extract_betas import load_expdesign, extract_betas_for_subject

df_ids = pd.read_csv(SELECTED_IDS)
nsd_ids = sorted(df_ids['nsd_id'].astype(int).unique().tolist())
print(f'Extracting betas for {len(nsd_ids)} images')

expdesign = load_expdesign(NSD_ROOT)
print('Loaded NSD experiment design')

In [ ]:
os.makedirs(BETA_DIR, exist_ok=True)

for sub_idx, subject in enumerate(SUBJECTS):
    out_path = os.path.join(BETA_DIR, f'{subject}_betas.npz')
    if os.path.exists(out_path):
        print(f'{subject}: already extracted, skipping')
        continue

    print(f'Extracting {subject}...')
    sub_betas = extract_betas_for_subject(
        subject=subject,
        subject_idx=sub_idx,
        nsd_ids=nsd_ids,
        expdesign=expdesign,
        nsd_root=NSD_ROOT,
        space=SPACE,
    )

    if not sub_betas:
        print(f'  WARNING: No betas for {subject}')
        continue

    ordered_ids = sorted(sub_betas.keys())
    beta_matrix = np.stack([sub_betas[nid] for nid in ordered_ids])
    np.savez_compressed(out_path, betas=beta_matrix, nsd_ids=np.array(ordered_ids))
    print(f'  Saved: {beta_matrix.shape} → {out_path}')

## Step 2: Voxelwise Regression

In [ ]:
from voxelwise_regression import build_design_matrix, fit_voxelwise_with_stats

df_scores = pd.read_csv(SCORES_FILE)
print(f'Loaded {len(df_scores)} image scores')

all_beta_maps = []
regressor_names = None

for subject in SUBJECTS:
    beta_file = os.path.join(BETA_DIR, f'{subject}_betas.npz')
    if not os.path.exists(beta_file):
        print(f'Skipping {subject}')
        continue

    data = np.load(beta_file)
    Y = data['betas']
    sub_nsd_ids = data['nsd_ids']

    # Align scores
    df_sub = df_scores[df_scores['nsd_id'].isin(sub_nsd_ids)].copy()
    df_sub = df_sub.set_index('nsd_id').loc[sub_nsd_ids].reset_index()
    valid_mask = df_sub['load_z'].notna() & df_sub['abstractness_z'].notna()
    df_sub = df_sub[valid_mask]
    Y = Y[valid_mask.values]

    X, regressor_names = build_design_matrix(df_sub)
    betas, t_stats, p_values = fit_voxelwise_with_stats(X, Y)

    # Save
    glm_path = os.path.join(BETA_DIR, f'{subject}_glm.npz')
    np.savez_compressed(glm_path, betas=betas, t_stats=t_stats, p_values=p_values,
                        regressor_names=regressor_names)

    all_beta_maps.append(betas)
    print(f'{subject}: X={X.shape}, Y={Y.shape}, betas={betas.shape}')

print(f'\nProcessed {len(all_beta_maps)} subjects')
print(f'Regressors: {regressor_names}')

## Step 3: Group-Level Analysis

In [ ]:
from voxelwise_regression import group_level_ttest

# Align voxel counts
min_voxels = min(b.shape[1] for b in all_beta_maps)
aligned = [b[:, :min_voxels] for b in all_beta_maps]

t_maps, p_maps, sig_mask = group_level_ttest(aligned, fdr_q=FDR_Q)

print(f'Group-level results ({len(aligned)} subjects):')
for i, name in enumerate(regressor_names):
    n_sig = sig_mask[i].sum()
    total = sig_mask[i].size
    pct = 100 * n_sig / total
    print(f'  {name}: {n_sig:,}/{total:,} voxels significant ({pct:.1f}%)')

# Save
os.makedirs('results/group', exist_ok=True)
np.savez_compressed('results/group/group_glm.npz',
                    t_maps=t_maps, p_maps=p_maps,
                    significant_mask=sig_mask,
                    regressor_names=regressor_names)

## Visualization

In [ ]:
# Plot group-level t-maps as flat vector plots
n_regressors = len(regressor_names)
fig, axes = plt.subplots(n_regressors, 1, figsize=(14, 3 * n_regressors))
if n_regressors == 1:
    axes = [axes]

for i, (name, ax) in enumerate(zip(regressor_names, axes)):
    t_data = t_maps[i]
    vmax = np.percentile(np.abs(t_data), 99)

    # Plot t-values as a 1D strip (pseudo-flatmap)
    n_vox = len(t_data)
    side = int(np.ceil(np.sqrt(n_vox)))
    padded = np.full(side * side, np.nan)
    padded[:n_vox] = t_data
    im = ax.imshow(padded.reshape(side, side), cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_title(f'{name} — Group t-map')
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='t-statistic', shrink=0.6)

plt.tight_layout()
plt.savefig('results/group/group_tmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/group/group_tmaps.png')

In [ ]:
# Summary: percentage of significant voxels per regressor
fig, ax = plt.subplots(figsize=(8, 5))
sig_pcts = [100 * sig_mask[i].sum() / sig_mask[i].size for i in range(n_regressors)]
bars = ax.bar(regressor_names, sig_pcts, color=['steelblue', 'mediumpurple', 'coral', 'gray', 'lightgray'][:n_regressors])
ax.set_ylabel('% Significant Voxels (FDR q<0.05)')
ax.set_title('Group-Level GLM: Significant Voxels per Regressor')
for bar, pct in zip(bars, sig_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('results/group/significant_voxels_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### Optional: Surface Projection (requires nilearn + fsaverage)

If you have the fsaverage surface meshes, you can project t-maps onto cortical surfaces.

In [ ]:
# Optional: project onto fsaverage surface using nilearn
try:
    from nilearn import datasets, plotting, surface

    fsavg = datasets.fetch_surf_fsaverage()

    for i, name in enumerate(regressor_names[:3]):  # load, abstract, interaction
        t_data = t_maps[i]
        # Split into hemispheres (assuming lh then rh concatenation)
        n_half = len(t_data) // 2
        lh_data = t_data[:n_half]
        rh_data = t_data[n_half:]

        # Pad/trim to fsaverage vertex count if needed
        n_fsavg = 163842  # fsaverage vertex count per hemi
        for hemi_name, hemi_data, surf_key in [
            ('left', lh_data, 'infl_left'),
            ('right', rh_data, 'infl_right')
        ]:
            padded = np.zeros(n_fsavg)
            padded[:min(len(hemi_data), n_fsavg)] = hemi_data[:n_fsavg]

            fig = plotting.plot_surf_stat_map(
                fsavg[surf_key], padded,
                hemi=hemi_name, view='lateral',
                title=f'{name} ({hemi_name})',
                colorbar=True, cmap='RdBu_r',
                threshold=2.0
            )
            plt.show()

except ImportError:
    print('nilearn not available — skipping surface visualization')
except Exception as e:
    print(f'Surface visualization failed: {e}')